# Chapter 4 Lab — Dependency Parsing

Newly written for this book (no usable existing lab). Loads a small pretrained spaCy pipeline,
visualizes dependency trees, and extracts subject-verb-object triples.

In [ ]:
import spacy
# python -m spacy download en_core_web_sm  (run once, see requirements.txt / README)
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy mode: en_core_web_sm")
except Exception as exc:
    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")
    print(f"spaCy mode: blank English fallback ({type(exc).__name__})")


## 1. Inspect a dependency parse

In [ ]:
doc = nlp("The cat sat on the mat.")
for tok in doc:
    print(f"{tok.text:<8}{tok.dep_:<10}head={tok.head.text:<8}pos={tok.pos_}")

In [ ]:
from spacy import displacy
displacy.render(doc, style="dep", jupyter=True)

## 2. Extract subject-verb-object triples

In [ ]:
def extract_svo(doc):
    triples = []
    for tok in doc:
        if tok.dep_ == "ROOT" and tok.pos_ == "VERB":
            subj = [c for c in tok.children if c.dep_ in ("nsubj", "nsubjpass")]
            obj = [c for c in tok.children if c.dep_ in ("dobj", "attr", "pobj")]
            if subj and obj:
                triples.append((subj[0].text, tok.text, obj[0].text))
    return triples

sentences = [
    "The cat sat on the mat.",
    "Maria gave Peter a book.",
    "The committee rejected the proposal.",
]
for s in sentences:
    print(s, "->", extract_svo(nlp(s)))

## 3. Same relation, different word order

English is fairly fixed SVO. Dependency labels (nsubj, dobj) stay meaningful even when surface
order changes across languages — that's the point of using relations instead of position.

In [ ]:
variants = ["The dog chased the ball.", "It was the ball that the dog chased."]
for s in variants:
    d = nlp(s)
    print(s)
    for tok in d:
        if tok.dep_ in ("nsubj", "dobj", "pobj"):
            print("  ", tok.dep_, "->", tok.text)

## Exercise

Run `extract_svo` on five sentences of your own, including at least one with a relative clause
or passive voice. Where does the simple rule-based extractor break, and why?